# Frameworks for fine-tuning

Now that we know more about what it means to fine-tune, we can see how to do it practically. Most modern deep learning developments are, one way or another, based on [Pytorch](https://pytorch.org). In principle, nothing would stop us from implementing a fine-tuning pipeline in pure Pytorch. However, there would be a lot of boilerplate code to be written; moreover, if we decided that our workload needs multiple GPUs, vanilla Pytorch would require some level of code rewriting. To avoid this, a number of frameworks that abstract over Pytorch have risen, such as [Pytorch lightning](https://lightning.ai/docs/pytorch/stable) and Hugging Face (HF) [accelerate](https://github.com/huggingface/accelerate) with the rest of the HF [stack](https://github.com/orgs/huggingface/repositories). 

Lightning is very generic and can be used to wrap any neural network architecture. The HF libraries, instead, are mainly meant to be used with transformers (such as LLMs) and provide a tight integration with the [hub](https://github.com/huggingface/huggingface_hub) for model downloading and sharing, [trl](https://github.com/huggingface/trl) for reinforcement learning, [tokenizers](https://github.com/huggingface/tokenizers) and [peft](https://github.com/huggingface/peft) for parameter-efficient finetuning (such as LoRA), among many others. Together, they constitute a tightly integrated ecosystem which simplifies life for both research and production for people working with LLMs. Thus, we will focus on its usage!

## The Hugging Face fine-tuning stack

The HF ecosystem is deliberately modular, and different libraries deal with different parts of the training workflow. For our fine-tuning example, we will deal with the following:

- Transformers: used to load the models, their tokenizers and relative configuration
- PEFT: used to implement parameter-efficient strategies, such as LoRA, QLoRA, adapters, etc.
- TRL: used to implement the actual training loop, implementing algorithms such as supervised fine-tuning (SFT) and some RL algorithms (DPO, PPO, GRPO, etc.)
- Accelerate: used to abstract the underlying hardware and parallelisation strategies (more on that later)

Our fine-tuning script will simply combine these building blocks :)

### Transformers: models and tokenizers
The [transformers](https://github.com/huggingface/transformers) library provides means to load pretrained models and tokenizers, as well as potentially managing their distribution to different hardware, the precision at which to load them, and lots more. In our case, let us use a small Qwen 1.5B model as base:
```python
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.bfloat16)
```
At this stage, we have just loaded the model, but we have not said anything about the training method nor distribution strategy. A small note on the `use_fast` parameter: some tokenizers have been implemented by the HF developers in Rust and reach very good performance. If such a tokenizer is available for the chosen model, `use_fast=True` will load it!

### PEFT: Parameter-Efficient Fine-Tuning

As discussed earlier, training all parameters of an LLM can be expensive, thus it is often advised to resort to techniques like (Q)LoRA. The `peft` library allows us to do it in the following way:
```python
from peft import LoraConfig, TaskType

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    task_type=TaskType.CAUSAL_LM,
    target_modules=['q_proj','k_proj','v_proj','o_proj','fc_in','fc_out'])
```

In this case, we are specifying all the modules/layer to which we want to apply LoRA (all the projections in this case). This was the historical way of prescribing the layers and requires knowledge of the model architecture itself. The model layers can be inspected with:
```python
for name, module in model.named_modules():
    print(name)
```
This will print all layers; we can also look at just the projection parts by filtering on the ones that have `"proj"` in the name.
However, nowadays people apply (Q)LoRA to all linear layers. This can be achieved by using `target_modules="all-linear"`, thus abstracting the model architecture away.

### TRL: The fine-tuning loop

TRL is the HF library that actually deals with the different training techniques: (supervised) finetuning (SFT), DPO, PPO, etc. In our case we want to do SFT, so we'll be using two objects: `SFTConfig` and `SFTTrainer`.

```python
training_args = SFTConfig(
    output_dir="./outputs",
    #optimisation
    learning_rate=2e-4,
    weight_decay=0.01,
    #batch sizes
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    #scheduling
    num_train_epochs=1,
    warmup_ratio=0.03,
    #logging
    logging_steps=10,
    #checkpointing
    save_strategy="epoch",
    #precision
    bf16=True,
    #packing
    packing=True
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
    peft_config=peft_config
)
```

The training can then be launched with `trainer.train()`. In the lines above, `packing=True` is an optimisation that packs several datapoints in one longer sequence to improve GPU utilisation. The `processing_class`, instead, is the "transformation" that the dataset undergoes when fed to the model; for pure text, that would be the tokeniser (like in this example), possibly followed by application of the chat template, padding, etc. For vision-language models or audio models, it can be a more generic `processor`.

### Accelerate: distributing models and data on the hardware and running

The role of Accelerate is to orchestrate the running of the training on the actual hardware, potentially dealing with distribution. Its main feature is the `Accelerator` object, which we are not using directly since `SFTTrainer` creates one behind the curtains.
The true power of `accelerate` lies in multi-GPU orchestration. Imagine that we have a very large dataset, and churning it through a single GPU would take a very long time, or that we have a model that is too big to fit on a single GPU and we want to "spread it" over multiple GPUs to finetune it. Two parallelisation strategies apply to each case respectively:
- DDP (Distributed Data Parallel): the model fits on one GPU, but we have a lot of data. DDP creates one copy of the model on each GPU and gives it a chunk of the data. Then at the end of each step, the gradients are collected from all the GPUs, an optimiser step is performed, and the updated weights are redistributed to all model copies
- FSDP (Fully-Sharded Data Parallel): the model is too big to fit on one GPU, thus it is spread (sharded) across the GPUs with a chunk of the data. Some complex orchestration makes the execution work.
There are also other more complex types of parallelism (tensor parallelism, pipeline parallelism) that are becoming more and more relevant. `Accelerate` lets us swap distribution strategies at launch time, without tweaking the training code, even across different nodes. Our training script can be launched as follows:

```bash
accelerate launch --multi_gpu --num_processes 4 train.py
```
This will automatically enable DDP across 4 GPUs. If we want to do FSDP, we just do the following:
```bash
accelerate launch --use_fsdp --num_processes 4 train.py
```

Usually, a more resilient way of prescribing different distribution strategies is by using some YAML files that contain all the options we want to pass, for example:

```yaml
compute_environment: LOCAL_MACHINE
distributed_type: MULTI_GPU
num_processes: 8
mixed_precision: bf16
```
for DDP and 
```yaml
compute_environment: LOCAL_MACHINE
distributed_type: FSDP
num_processes: 8
mixed_precision: bf16
fsdp_config:
fsdp_auto_wrap_policy: TRANSFORMER_BASED_WRAP
fsdp_sharding_strategy: FULL_SHARD
fsdp_state_dict_type: SHARDED_STATE_DICT
``` 
for FSDP. The training can then be launched with `accelerate launch --config_file ddp.yaml train.py`. These YAML files can also be created interactively using a wizard with `accelerate config`. More info can be found in the [documentation](https://huggingface.co/docs/accelerate/main/en/package_reference/cli#accelerate-config). 

## Practical example

We want to fine-tune a small Qwen2.5-1.5B-Instruct [model](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct) on the standard [Dolly](https://huggingface.co/datasets/databricks/databricks-dolly-15k) instruction-following dataset. The bulk of the training code is a sum of the snippets we've shown until now, with just one addition in the beginning: the dolly datasets contains three fields (input, instruction and output). Input and instruction are both user-inputs (one is a query, one is additional context), but not all rows contain both of them. So we do a quick preprocessing step where we check whether either of them exists, and if both are there, we merge them.

In [ ]:
%%writefile train.py

import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, TaskType
from trl import SFTConfig, SFTTrainer

# Dataset creation and preprocessing 

dataset = load_dataset(
    "json",
    data_files={
        "train": "data/dolly_train.jsonl",
        "validation": "data/dolly_val.jsonl"
    }
)

def to_conversation(example):

    instruction = (example.get("instruction") or "").strip()
    context = (example.get("input") or "").strip()
    answer = (example.get("output") or "").strip()

    if instruction and context:
        user_content = (
            f"{instruction}\n\n"
            f"Context:\n{context}"
        )

    elif instruction:
        user_content = instruction

    elif context:
        user_content = context

    else:
        user_content = ""

    return {
        "messages": [
            {
                "role": "user",
                "content": user_content,
            },
            {
                "role": "assistant",
                "content": answer,
            },
        ]
    }


dataset = dataset.map(
    to_conversation,
    remove_columns=dataset["train"].column_names,
)

train_dataset = dataset["train"]
eval_dataset = dataset["validation"]

# Model instantiation 

model_name = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16)

peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    target_modules="all-linear",
    r=16,
    lora_alpha=32,
    lora_dropout=0.05
)

training_args = SFTConfig(
    output_dir="./outputs",
    num_train_epochs=1,
    learning_rate=2e-4,
    warmup_ratio=0.03,
    weight_decay=0.01,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=8,
    bf16=True,
    logging_steps=10,
    save_strategy="epoch",
    packing=True,
    eval_strategy="steps",
    eval_steps=10,
    max_length=512
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    peft_config=peft_config
)

trainer.model.print_trainable_parameters()

trainer.train()

trainer.save_model("./outputs/final")